In [1]:
import pandas as pd
from datetime import datetime

In [2]:
# helper function for reading datatset
def read_data(file_path):
    df = pd.read_csv(file_path)
    df['Date'] = pd.to_datetime(df['Date'], format='%Y-%m-%d') # convert it to datatime

    return df



In [5]:

df_train = read_data('/Users/jeonginn/Desktop/ML/kospi_train.csv')
df_test = read_data('/Users/jeonginn/Desktop/ML/kospi_test.csv')

len(df_train), len(df_test)


(986, 244)

In [6]:

# the training dataset has daily KOSPI index from 2019 to 2022
df_train


,Date,Open,Low,High,Close,Volume
0,2019-01-02,2050.550049,2004.270020,2053.449951,2010.000000,326400
1,2019-01-03,2011.810059,1991.650024,2014.719971,1993.699951,428000
2,2019-01-04,1992.400024,1984.530029,2011.560059,2010.250000,409000
3,2019-01-07,2034.239990,2030.900024,2048.060059,2037.099976,440200
4,2019-01-08,2038.680054,2023.589966,2042.699951,2025.270020,397800
...,...,...,...,...,...,...
981,2022-12-23,2325.860107,2311.899902,2333.080078,2313.689941,367000
982,2022-12-26,2312.540039,2304.199951,2321.919922,2317.139893,427600
983,2022-12-27,2327.520020,2321.479980,2335.989990,2332.790039,448300
984,2022-12-28,2296.449951,2276.899902,2296.449951,2280.449951,405700


In [7]:

# the test dataset has daily KOSPI index in 2023
df_test


,Date,Open,Low,High,Close,Volume
0,2023-01-02,2249.949951,2222.370117,2259.879883,2225.669922,346100
1,2023-01-03,2230.979980,2180.669922,2230.979980,2218.679932,410000
2,2023-01-04,2205.979980,2198.820068,2260.060059,2255.979980,412700
3,2023-01-05,2268.199951,2252.969971,2281.389893,2264.649902,430800
4,2023-01-06,2253.399902,2253.270020,2300.620117,2289.969971,398300
...,...,...,...,...,...,...
239,2023-12-21,2598.370117,2587.159912,2610.810059,2600.020020,578300
240,2023-12-22,2617.719971,2599.510010,2621.370117,2599.510010,466000
241,2023-12-26,2609.439941,2594.649902,2612.139893,2602.590088,439500
242,2023-12-27,2599.350098,2590.080078,2613.500000,2613.500000,349700


In [8]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np


# 2. 타깃 열 생성 (다음날 종가)
df_train['Target'] = df_train['Close'].shift(-1)
df_train = df_train.dropna().reset_index(drop=True)

# 3. 훈련 데이터 준비
features = ['Open', 'Low', 'High', 'Close', 'Volume']
X_train = df_train[features]
y_train = df_train['Target']

# 4. 모델 훈련
model = LinearRegression()
model.fit(X_train, y_train)

# 5. 테스트 데이터 준비
# 테스트셋은 다음날 종가가 없으므로 직접 예측만 수행
X_test = df_test[features]
predictions = model.predict(X_test)

# 6. 결과 출력
df_test['Predicted_Close'] = predictions
print(df_test[['Date', 'Close', 'Predicted_Close']].head())

# (선택) 실제 다음날 종가가 있다면 평가도 가능
# 예: y_test = df_test['Target'] 가 존재할 경우
# rmse = mean_squared_error(y_test, predictions, squared=False)
# print("Test RMSE:", rmse)


        Date        Close  Predicted_Close
0 2023-01-02  2225.669922      2228.272777
1 2023-01-03  2218.679932      2215.411903
2 2023-01-04  2255.979980      2258.163145
3 2023-01-05  2264.649902      2265.067957
4 2023-01-06  2289.969971      2292.885524


In [9]:
# 원본 훈련 데이터 복사
df = pd.read_csv("kospi_train.csv")
df['Target'] = df['Close'].shift(-1)
df = df.dropna().reset_index(drop=True)

# 이동 평균 (5일)
df['MA_5'] = df['Close'].rolling(window=5).mean()

# 변동성
df['Range'] = df['High'] - df['Low']

# 종가 대비 고가/저가 비율
df['High_Close_Ratio'] = (df['High'] - df['Close']) / df['Close']
df['Close_Low_Ratio'] = (df['Close'] - df['Low']) / df['Close']

# 거래량 변화율
df['Volume_Change'] = df['Volume'] / df['Volume'].shift(1)

# 결측치 제거
df = df.dropna().reset_index(drop=True)

# 새롭게 추가된 피처
features = ['Open', 'Low', 'High', 'Close', 'Volume',
            'MA_5', 'Range', 'High_Close_Ratio', 'Close_Low_Ratio', 'Volume_Change']

X = df[features]
y = df['Target']


In [10]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

# 데이터 불러오기
df = pd.read_csv("kospi_train.csv")
df['Target'] = df['Close'].shift(-1)

# 추가 피처 생성
df['MA_5'] = df['Close'].rolling(window=5).mean()
df['Range'] = df['High'] - df['Low']
df['High_Close_Ratio'] = (df['High'] - df['Close']) / df['Close']
df['Close_Low_Ratio'] = (df['Close'] - df['Low']) / df['Close']
df['Volume_Change'] = df['Volume'] / df['Volume'].shift(1)

df = df.dropna().reset_index(drop=True)

# 피처 및 타깃 설정
features = ['Open', 'Low', 'High', 'Close', 'Volume',
            'MA_5', 'Range', 'High_Close_Ratio', 'Close_Low_Ratio', 'Volume_Change']
X = df[features]
y = df['Target']

# 모델 정의
models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42)
}

# 교차검증
tscv = TimeSeriesSplit(n_splits=5)
results = {}

for name, model in models.items():
    rmses = []
    for train_idx, val_idx in tscv.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model.fit(X_train, y_train)
        preds = model.predict(X_val)
        rmse = mean_squared_error(y_val, preds, squared=False)
        rmses.append(rmse)
    
    results[name] = {
        "mean_rmse": np.mean(rmses),
        "std_rmse": np.std(rmses)
    }

# 결과 출력
for model_name, res in results.items():
    print(f"{model_name}: 평균 RMSE = {res['mean_rmse']:.2f}, 표준편차 = {res['std_rmse']:.2f}")


LinearRegression: 평균 RMSE = 33.20, 표준편차 = 3.63
RandomForest: 평균 RMSE = 136.90, 표준편차 = 94.83


LinearRegression: 평균 RMSE = 33.20, 표준편차 = 3.63
RandomForest: 평균 RMSE = 136.90, 표준편차 = 94.83
